In [1]:
pip install anthropic pillow

  Using cached anthropic-0.86.0-py3-none-any.whl.metadata (3.0 kB)
  Using cached pillow-12.1.1-cp314-cp314-macosx_11_0_arm64.whl.metadata (8.8 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached docstring_parser-0.17.0-py3-none-any.whl.metadata (3.5 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached certifi-2026.2.25-py3-none-any.whl.metadata (2.5 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
Using cached anthropic-0.86.0-py3-none-any.whl (46

In [13]:
!pip install pandas openpyxl

  Using cached pandas-3.0.1-cp314-cp314-macosx_11_0_arm64.whl.metadata (79 kB)
  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached numpy-2.4.3-cp314-cp314-macosx_14_0_arm64.whl.metadata (6.6 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached pandas-3.0.1-cp314-cp314-macosx_11_0_arm64.whl (9.9 MB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached numpy-2.4.3-cp314-cp314-macosx_14_0_arm64.whl (5.2 MB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [openpyxl]3/4 [openpyxl]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [10]:
# تست برای دیدن لیست مدل‌های در دسترس شما
models = client.models.list()
for model in models.data:
    print(model.id)

claude-sonnet-4-6
claude-opus-4-6
claude-opus-4-5-20251101
claude-haiku-4-5-20251001
claude-sonnet-4-5-20250929
claude-opus-4-1-20250805
claude-opus-4-20250514
claude-sonnet-4-20250514


In [ ]:
import base64
import json
import re
import pandas as pd
from dotenv import load_dotenv      
import os
from anthropic import Anthropic

# Initialize Anthropic client
load_dotenv()
client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

def extract_table_data(image_path, image_media_type="image/gif"):
    image_data = encode_image(image_path)
    
    # IMPROVED EXPERT PROMPT
    system_instruction = """You are an expert in digitizing historical hydrological yearbooks. 
    Your goal is to extract the 'PORTATE MEDIE GIORNALIERE' table with 100% accuracy.
    
    CRITICAL RULES:
    1. EXTRACT ONLY the main daily discharge table (columns: GIORNO, Gennaio to Dicembre).
    2. IGNORE all other summary tables like 'ELEMENTI CARATTERISTICI' or 'SCALA NUMERICA'.
    3. DATA FORMAT: Return a JSON array of exactly 31 objects (one for each day).
    4. MISSING VALUES: If a cell contains a dash '-', a dot '.', or is empty, use null.
    5. NUMBERS: Maintain decimal precision. Some numbers might be bold—treat them as normal numbers.
    6. OUTPUT: Return ONLY raw JSON. No markdown, no explanations."""

    user_instruction = "Extract the daily discharge data from this image for all months and all 31 days. Ensure each row corresponds to the 'GIORNO' column."

    message = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=4000, 
        temperature=0, # Keeps the model focused and deterministic
        system=system_instruction,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "source": {
                            "type": "base64",
                            "media_type": image_media_type,
                            "data": image_data,
                        },
                    },
                    {
                        "type": "text",
                        "text": user_instruction
                    }
                ],
            }
        ],
    )
    return message.content[0].text

# --- Execution Logic ---
my_file = "../test_pictures/Q_Bari_1969_26.gif"

print("Starting expert extraction...")
raw_output = extract_table_data(my_file)

# Cleaning and Saving
clean_output = re.search(r'\[.*\]', raw_output, re.DOTALL)
if clean_output:
    try:
        json_data = json.loads(clean_output.group(0))
        df = pd.DataFrame(json_data)
        df.to_excel("expert_extraction_result.xlsx", index=False)
        print("Success! Expert extraction saved to expert_extraction_result.xlsx")
    except Exception as e:
        print(f"JSON Error: {e}")
else:
    print("Could not find valid data.")

Starting expert extraction...
Success! Expert extraction saved to expert_extraction_result.xlsx


In [ ]:
import base64
import json
import re
import pandas as pd
import os
from dotenv import load_dotenv  
from anthropic import Anthropic

# Initialize Anthropic client
load_dotenv()
client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

def extract_table_data(image_path, image_media_type="image/gif"):
    image_data = encode_image(image_path)
    
    # IMPROVED EXPERT PROMPT
    system_instruction = """You are an elite Data Auditor specialized in historical hydrological yearbooks. 
    Your mission is to transcribe the daily discharge table with 100% precision, regardless of naming variations.

    ### 1. SEMANTIC IDENTIFICATION (STRUCTURE):
    - IDENTIFY THE DAY COLUMN: Look for a column containing numbers 1 through 31. It might be labeled 'Giorno', 'Giorni', 'G.', or have NO label at all. This is your Index.
    - IDENTIFY MONTH COLUMNS: Look for 12 columns representing months (Gennaio to Dicembre). Labels might vary (e.g., 'Genn.', 'Febbr.', or just 'I, II, III...'). 
    - TABLE BOUNDARIES: Focus ONLY on the grid of 31 rows by 12 months. Ignore summary rows like 'Media', 'Massima', or 'Totale' at the bottom.

    ### 2. MANDATORY TWO-PASS AUDIT:

    *PASS 1: VERTICAL INTEGRITY CHECK (COLUMN-BY-COLUMN)*
    For each identified month column:
    a) Count the actual numeric entries (ignore dots, dashes, or empty spaces).
    b) Cross-reference the count with the calendar days of that month (Jan=31, Feb=28/29, Mar=31, Apr=30, etc.).
    c) IF THE COUNT IS LOW: Re-scan the column vertically. Historical prints often have compressed rows where numbers drift upward. Use the bottom-up and top-down alignment to ensure every number is assigned to its CORRECT 'Giorno' (Day).
    d) NO INVENTIONS: If a cell is truly empty or contains a dash/dot, use 'null'. Do NOT guess.

    *PASS 2: HORIZONTAL ALIGNMENT*
    - Ensure the value for 'Day 15' in 'April' is strictly aligned with the number 15 in the Day column.

    ### 3. OUTPUT RULES:
    - Treat brackets [ ], parentheses ( ), or asterisks * as noise. Extract only the number (e.g., "[12.5]*" -> 12.5).
    - Return a JSON list of 31 objects. 
    - Each object MUST use these keys: "Giorno", "Gen", "Feb", "Mar", "Apr", "Mag", "Giu", "Lug", "Ago", "Set", "Ott", "Nov", "Dic".
    - Map the extracted month columns to these keys regardless of how they are labeled in the image.

    Strictly return ONLY the raw JSON array. No conversational text.
    ### 4. CRITICAL: Do not perform any statistical guessing. If a digit looks like an 8, verify the closed loops. Examine the pixel structure of the digit's head and tail separately before confirming the number"""

    user_instruction = "Extract the daily discharge data from this image for all months and all 31 days. Ensure each row corresponds to the 'GIORNO' column."

    message = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=4000, 
        temperature=0, # Keeps the model focused and deterministic
        system=system_instruction,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "source": {
                            "type": "base64",
                            "media_type": image_media_type,
                            "data": image_data,
                        },
                    },
                    {
                        "type": "text",
                        "text": user_instruction
                    }
                ],
            }
        ],
    )
    return message.content[0].text

# --- Execution Logic ---
my_file = "../test_pictures/Q_Cagliari_1924_66.gif"

print("Starting expert extraction...")
raw_output = extract_table_data(my_file)

# Cleaning and Saving
clean_output = re.search(r'\[.*\]', raw_output, re.DOTALL)
if clean_output:
    try:
        json_data = json.loads(clean_output.group(0))
        df = pd.DataFrame(json_data)
        df.to_excel("expert_extraction_result.xlsx", index=False)
        print("Success! Expert extraction saved to expert_extraction_result.xlsx")
    except Exception as e:
        print(f"JSON Error: {e}")
else:
    print("Could not find valid data.")

Starting expert extraction...
Success! Expert extraction saved to expert_extraction_result.xlsx


In [6]:
import openpyxl

!pip install opencv-python numpy pandas openpyxl anthropic


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 MB 30.4 MB/s  0:00:01 eta 0:00:01

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
import base64
import json
import re
import cv2
import numpy as np
import os
from dotenv import load_dotenv  
import pandas as pd
from anthropic import Anthropic

# --- تنظیمات ---
# کلید اختصاصی خودت را اینجا وارد کن
API_KEY = os.getenv("ANTHROPIC_API_KEY")
load_dotenv()
client = Anthropic(api_key=API_KEY)


def preprocess_image(image_path):
    """
    بهبود کیفیت تصویر برای جلوگیری از اشتباه گرفتن اعداد (مثل 8 با 9)
    """
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return image_path

    # بزرگ‌نمایی 2 برابری برای باز شدن پیکسل‌های اعداد فشرده
    img = cv2.resize(img, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)

    # افزایش کنتراست (CLAHE) برای شفاف کردن مرز اعداد
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    img = clahe.apply(img)

    # ذخیره تصویر بهبود یافته (خروجی همیشه JPEG است)
    temp_path = "temp_enhanced_image.jpg"
    cv2.imwrite(temp_path, img)
    return temp_path

def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

def extract_table_data(image_path):
    # مرحله ۱: پیش‌پردازش تصویر
    processed_path = preprocess_image(image_path)
    image_data = encode_image(processed_path)
    
    # نکته حیاتی: چون خروجی preprocess همیشه JPG است، مدیا تایپ باید jpeg باشد
    image_media_type = "image/jpeg"

    # --- ابر پرامپت تحلیلگر (Ultimate Semantic Auditor) ---
    system_instruction = """You are an elite Data Auditor specialized in historical hydrological yearbooks. 
    Your mission is to transcribe the daily discharge table with 100% precision, regardless of naming variations.

    ### 1. SEMANTIC IDENTIFICATION (STRUCTURE):
    - IDENTIFY THE DAY COLUMN: Look for a column containing numbers 1 through 31. It might be labeled 'Giorno', 'Giorni', 'G.', or have NO label at all. This is your Index.
    - IDENTIFY MONTH COLUMNS: Look for 12 columns representing months (Gennaio to Dicembre). Labels might vary (e.g., 'Genn.', 'Febbr.', or just 'I, II, III...'). 
    - TABLE BOUNDARIES: Focus ONLY on the grid of 31 rows by 12 months. Ignore summary rows like 'Media', 'Massima', or 'Totale' at the bottom.

    ### 2. MANDATORY TWO-PASS AUDIT:

    *PASS 1: VERTICAL INTEGRITY CHECK (COLUMN-BY-COLUMN)*
    For each identified month column:
    a) Count the actual numeric entries (ignore dots, dashes, or empty spaces).
    b) Cross-reference the count with the calendar days of that month (Jan=31, Feb=28/29, Mar=31, Apr=30, etc.).
    c) IF THE COUNT IS LOW: Re-scan the column vertically. Historical prints often have compressed rows where numbers drift upward. Use the bottom-up and top-down alignment to ensure every number is assigned to its CORRECT 'Giorno' (Day).
    d) NO INVENTIONS: If a cell is truly empty or contains a dash/dot, use 'null'. Do NOT guess.

    *PASS 2: HORIZONTAL ALIGNMENT*
    - Ensure the value for 'Day 15' in 'April' is strictly aligned with the number 15 in the Day column.

    ### 3. OUTPUT RULES:
    - Treat brackets [ ], parentheses ( ), or asterisks * as noise. Extract only the number (e.g., "[12.5]*" -> 12.5).
    - Return a JSON list of 31 objects. 
    - Each object MUST use these keys: "Giorno", "Gen", "Feb", "Mar", "Apr", "Mag", "Giu", "Lug", "Ago", "Set", "Ott", "Nov", "Dic".
    - Map the extracted month columns to these keys regardless of how they are labeled in the image.

    Strictly return ONLY the raw JSON array. No conversational text.
    ### 4. CRITICAL: Do not perform any statistical guessing. If a digit looks like an 8, verify the closed loops. Examine the pixel structure of the digit's head and tail separately before confirming the number"""

    try:
        message = client.messages.create(
            model="claude-sonnet-4-6", 
            max_tokens=4000,
            temperature=0,
            system=system_instruction,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "image",
                            "source": {"type": "base64", "media_type": image_media_type, "data": image_data}
                        },
                        {"type": "text", "text": "Extract the daily discharge data using the Semantic Auditor Protocol."}
                    ]
                }
            ]
        )
        return message.content[0].text
    finally:
        # حذف فایل موقت بعد از اتمام کار
        if os.path.exists("temp_enhanced_image.jpg"):
            os.remove("temp_enhanced_image.jpg")

# --- بخش اجرا ---
my_file = "../test_pictures/Q_Cagliari_1933_91.gif"

if os.path.exists(my_file):
    print(f"Starting expert extraction for: {my_file}...")
    try:
        raw_output = extract_table_data(my_file)

        # استخراج جی‌سون از پاسخ
        json_match = re.search(r'\[\s*{.*}\s*\]', raw_output, re.DOTALL)
        if json_match:
            data = json.loads(json_match.group(0))
            df = pd.DataFrame(data)
            
            # مرتب‌سازی ستون‌ها
            cols = ["Giorno", "Gen", "Feb", "Mar", "Apr", "Mag", "Giu", "Lug", "Ago", "Set", "Ott", "Nov", "Dic"]
            df = df.reindex(columns=cols)
            
            output_name = "expert_extraction_result2.xlsx"
            df.to_excel(output_name, index=False)
            print(f"Success! Data saved to {output_name}")
        else:
            print("Could not find valid JSON. Raw output:")
            print(raw_output)
    except Exception as e:
        print(f"An error occurred: {e}")
else:
    print(f"File {my_file} not found.")

Starting expert extraction for: ../test_pictures/Q_Cagliari_1933_91.gif...
Success! Data saved to expert_extraction_result2.xlsx


In [ ]:
import base64
import json
import re
import pandas as pd
import os
from dotenv import load_dotenv  
from anthropic import Anthropic

# Initialize Anthropic client
load_dotenv()
client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

def extract_table_data(image_path, image_media_type="image/gif"):
    image_data = encode_image(image_path)
    
    # IMPROVED EXPERT PROMPT
    system_instruction = """You are an elite Data Auditor specialized in historical hydrological yearbooks. 
    Your mission is to transcribe the daily discharge table with 100% precision, regardless of naming variations.

    ### 1. SEMANTIC IDENTIFICATION (STRUCTURE):
    - IDENTIFY THE DAY COLUMN: Look for a column containing numbers 1 through 31. It might be labeled 'Giorno', 'Giorni', 'G.', or have NO label at all. This is your Index.
    - IDENTIFY MONTH COLUMNS: Look for 12 columns representing months (Gennaio to Dicembre). Labels might vary (e.g., 'Genn.', 'Febbr.', or just 'I, II, III...'). 
    - TABLE BOUNDARIES: Focus ONLY on the grid of 31 rows by 12 months. Ignore summary rows like 'Media', 'Massima', or 'Totale' at the bottom.

    ### 2. MANDATORY TWO-PASS AUDIT:

    *PASS 1: VERTICAL INTEGRITY CHECK (COLUMN-BY-COLUMN)*
    For each identified month column:
    a) Count the actual numeric entries (ignore dots, dashes, or empty spaces).
    b) Cross-reference the count with the calendar days of that month (Jan=31, Feb=28/29, Mar=31, Apr=30, etc.).
    c) IF THE COUNT IS LOW: Re-scan the column vertically. Historical prints often have compressed rows where numbers drift upward. Use the bottom-up and top-down alignment to ensure every number is assigned to its CORRECT 'Giorno' (Day).
    d) NO INVENTIONS: If a cell is truly empty or contains a dash/dot, use 'null'. Do NOT guess.

    *PASS 2: HORIZONTAL ALIGNMENT*
    - Ensure the value for 'Day 15' in 'April' is strictly aligned with the number 15 in the Day column.

    ### 3. OUTPUT RULES:
    - Treat brackets [ ], parentheses ( ), or asterisks * as noise. Extract only the number (e.g., "[12.5]*" -> 12.5).
    - Return a JSON list of 31 objects. 
    - Each object MUST use these keys: "Giorno", "Gen", "Feb", "Mar", "Apr", "Mag", "Giu", "Lug", "Ago", "Set", "Ott", "Nov", "Dic".
    - Map the extracted month columns to these keys regardless of how they are labeled in the image.

    Strictly return ONLY the raw JSON array. No conversational text.
    ### 4. CRITICAL: Do not perform any statistical guessing. If a digit looks like an 8, verify the closed loops. Examine the pixel structure of the digit's head and tail separately before confirming the number"""

    user_instruction = "Extract the daily discharge data from this image for all months and all 31 days. Ensure each row corresponds to the 'GIORNO' column."

    message = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=4000, 
        temperature=0, # Keeps the model focused and deterministic
        system=system_instruction,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "source": {
                            "type": "base64",
                            "media_type": image_media_type,
                            "data": image_data,
                        },
                    },
                    {
                        "type": "text",
                        "text": user_instruction
                    }
                ],
            }
        ],
    )
    return message.content[0].text

# --- Execution Logic ---
my_file = "../test_pictures/Q_Cagliari_1933_91.gif"

print("Starting expert extraction...")
raw_output = extract_table_data(my_file)

# Cleaning and Saving
clean_output = re.search(r'\[.*\]', raw_output, re.DOTALL)
if clean_output:
    try:
        json_data = json.loads(clean_output.group(0))
        df = pd.DataFrame(json_data)
        df.to_excel("expert_extraction_result2_2.xlsx", index=False)
        print("Success! Expert extraction saved to expert_extraction_result2_2.xlsx")
    except Exception as e:
        print(f"JSON Error: {e}")
else:
    print("Could not find valid data.")

Starting expert extraction...
Success! Expert extraction saved to expert_extraction_result2_2.xlsx


In [ ]:
import base64
import json
import re
import os
from dotenv import load_dotenv  
import pandas as pd
from anthropic import Anthropic

# Initialize Anthropic client
load_dotenv()
API_KEY = os.getenv("ANTHROPIC_API_KEY")
client = Anthropic(api_key=API_KEY)

def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

def extract_table_data(image_path, image_media_type="image/gif"):
    image_data = encode_image(image_path)
    
    # IMPROVED EXPERT PROMPT
    system_instruction = """You are an elite Data Auditor specialized in historical hydrological yearbooks. 
    Your mission is to transcribe the daily discharge table with 100% precision, regardless of naming variations.

    ### 1. SEMANTIC IDENTIFICATION (STRUCTURE):
    - IDENTIFY THE DAY COLUMN: Look for a column containing numbers 1 through 31. It might be labeled 'Giorno', 'Giorni', 'G.', or have NO label at all. This is your Index.
    - IDENTIFY MONTH COLUMNS: Look for 12 columns representing months (Gennaio to Dicembre). Labels might vary (e.g., 'Genn.', 'Febbr.', or just 'I, II, III...'). 
    - TABLE BOUNDARIES: Focus ONLY on the grid of 31 rows by 12 months. Ignore summary rows like 'Media', 'Massima', or 'Totale' at the bottom.

    ### 2. MANDATORY TWO-PASS AUDIT:

    *PASS 1: VERTICAL INTEGRITY CHECK (COLUMN-BY-COLUMN)*
    For each identified month column:
    a) Count the actual numeric entries (ignore dots, dashes, or empty spaces).
    b) Cross-reference the count with the calendar days of that month (Jan=31, Feb=28/29, Mar=31, Apr=30, etc.).
    c) IF THE COUNT IS LOW: Re-scan the column vertically. Historical prints often have compressed rows where numbers drift upward. Use the bottom-up and top-down alignment to ensure every number is assigned to its CORRECT 'Giorno' (Day).
    d) NO INVENTIONS: If a cell is truly empty or contains a dash/dot, use 'null'. Do NOT guess.

    *PASS 2: HORIZONTAL ALIGNMENT*
    - Ensure the value for 'Day 15' in 'April' is strictly aligned with the number 15 in the Day column.

    ### 3. OUTPUT RULES:
    - Treat brackets [ ], parentheses ( ), or asterisks * as noise. Extract only the number (e.g., "[12.5]*" -> 12.5).
    - Return a JSON list of 31 objects. 
    - Each object MUST use these keys: "Giorno", "Gen", "Feb", "Mar", "Apr", "Mag", "Giu", "Lug", "Ago", "Set", "Ott", "Nov", "Dic".
    - Map the extracted month columns to these keys regardless of how they are labeled in the image.

    Strictly return ONLY the raw JSON array. No conversational text.
    ### 4. CRITICAL: Do not perform any statistical guessing. If a digit looks like an 8, verify the closed loops. Examine the pixel structure of the digit's head and tail separately before confirming the number.
    ### 5. SPECIAL DIGIT DISCRIMINATION: Pay close attention to the digit '2'. In this typeface, the top hook of the '2' may appear nearly closed, resembling a '9'. However, look at the base: a '2' has a flat, horizontal base, while a '9' would have a vertical or curved tail. If you see a flat horizontal line at the bottom of a digit that has a closed-looking top, classify it as '2', not '9'."""

    

    user_instruction = "Extract the daily discharge data from this image for all months and all 31 days. Ensure each row corresponds to the 'GIORNO' column."

    message = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=4000, 
        temperature=0, # Keeps the model focused and deterministic
        system=system_instruction,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "source": {
                            "type": "base64",
                            "media_type": image_media_type,
                            "data": image_data,
                        },
                    },
                    {
                        "type": "text",
                        "text": user_instruction
                    }
                ],
            }
        ],
    )
    return message.content[0].text

# --- Execution Logic ---
my_file = "../test_pictures/Q_Cagliari_1933_91.gif"

print("Starting expert extraction...")
raw_output = extract_table_data(my_file)

# Cleaning and Saving
clean_output = re.search(r'\[.*\]', raw_output, re.DOTALL)
if clean_output:
    try:
        json_data = json.loads(clean_output.group(0))
        df = pd.DataFrame(json_data)
        df.to_excel("expert_extraction_result2_3.xlsx", index=False)
        print("Success! Expert extraction saved to expert_extraction_result2_3.xlsx")
    except Exception as e:
        print(f"JSON Error: {e}")
else:
    print("Could not find valid data.")

Starting expert extraction...
Success! Expert extraction saved to expert_extraction_result2_3.xlsx


In [31]:
import pandas as pd

# ۱. بارگذاری فایلی که توسط کد Claude ساخته شده
df_claude = pd.read_excel("expert_extraction_result2_3.xlsx")

# ۲. تبدیل نام ماه‌های ایتالیایی به عدد برای ستون دوم
month_map = {
    'Gen': 1, 'Feb': 2, 'Mar': 3, 'Apr': 4, 'Mag': 5, 'Giu': 6,
    'Lug': 7, 'Ago': 8, 'Set': 9, 'Ott': 10, 'Nov': 11, 'Dic': 12
}

# ۳. عملیات تبدیل جدول (Wide) به لیست (Long)
df_melted = df_claude.melt(id_vars=['Giorno'], var_name='Month_Name', value_name='Value')

# ۴. ساخت ستون‌های مورد نظر شما
df_final = pd.DataFrame()
df_final['Year'] = 1933  # ستون اول
df_final['Month'] = df_melted['Month_Name'].map(month_map) # ستون دوم: ماه
df_final['Day'] = df_melted['Giorno'] # ستون سوم: روز
df_final['Value'] = df_melted['Value'] # ستون چهارم: مقدار

# ۵. تبدیل مقادیر به عدد و جایگزینی NaNها با عدد ۰
# ابتدا مقادیر غیرعددی را به NaN تبدیل می‌کنیم (مثل >>)
df_final['Value'] = pd.to_numeric(df_final['Value'], errors='coerce')
# حالا تمام NaNها (چه خالی بوده‌ها و چه خطاها) را با 0 پر می‌کنیم
df_final['Value'] = df_final['Value'].fillna(0)

# مرتب‌سازی بر اساس ماه و روز
df_final = df_final.sort_values(by=['Month', 'Day'])

# ۶. ذخیره خروجی نهایی به سبک فایل Manual (بدون هدر و ایندکس)
output_filename = "claude_reformatted_to_manual.xlsx"
df_final.to_excel(output_filename, index=False, header=False)

print(f"تموم شد! فایل جدید با فرمت ۴ ستونه در '{output_filename}' ذخیره شد.")
print(df_final.head(10)) # نمایش ۱۰ سطر اول برای تست

تموم شد! فایل جدید با فرمت ۴ ستونه در 'claude_reformatted_to_manual.xlsx' ذخیره شد.
   Year  Month  Day  Value
0   NaN      1    1   0.52
1   NaN      1    2   0.44
2   NaN      1    3   0.44
3   NaN      1    4   0.44
4   NaN      1    5   0.36
5   NaN      1    6   0.36
6   NaN      1    7   0.31
7   NaN      1    8   0.31
8   NaN      1    9   0.31
9   NaN      1   10   0.31


In [32]:
import pandas as pd
import numpy as np

# ۱. بارگذاری فایل‌ها
# فایل دستی (Manual)
df_manual = pd.read_excel('../test_pictures/Q_Cagliari_1933_91.xlsx', header=None)
# فایل کلاود که با کد قبلی اصلاح کردید (reformatted)
df_claude = pd.read_excel('claude_reformatted_to_manual.xlsx', header=None)

# ۲. حذف ستون اول از هر دو (Ignore First Column)
# در هر دو فایل ستون 0 (سال) حذف می‌شود. ستون‌های باقی‌مانده: [ماه، روز، مقدار]
df_man_3cols = df_manual.iloc[:, 1:4].copy()
df_cld_3cols = df_claude.iloc[:, 1:4].copy()

# نام‌گذاری ستون‌ها برای جفت شدن درست
df_man_3cols.columns = ['Month', 'Day', 'Value_Man']
df_cld_3cols.columns = ['Month', 'Day', 'Value_Cld']

# ۳. فیلتر کردن فایل دستی برای سال ۱۹۳۳ (چون فایل کلاود فقط یک سال است)
df_man_1933 = df_man_3cols[df_manual.iloc[:, 0] == 1933].copy()

# ۴. ادغام (Merge) بر اساس ماه و روز
# این کار تضمین می‌کند که روز ۱ ماه ۱ در فایل اول با روز ۱ ماه ۱ در فایل دوم مقایسه شود
comparison = pd.merge(df_man_1933, df_cld_3cols, on=['Month', 'Day'])

# ۵. محاسبه مطابقت (Accuracy)
# استفاده از تلورانس کوچک برای جلوگیری از خطای اعشاری
comparison['Is_Match'] = np.isclose(
    comparison['Value_Man'].astype(float), 
    comparison['Value_Cld'].astype(float), 
    atol=1e-3
)

total_points = len(comparison)
matches = comparison['Is_Match'].sum()
accuracy = (matches / total_points) * 100 if total_points > 0 else 0

# ۶. چاپ خروجی
print(f"--- گزارش نهایی مقایسه ---")
print(f"تعداد کل روزهای مقایسه شده: {total_points}")
print(f"تعداد مطابقت‌های صحیح: {matches}")
print(f"دقت نهایی استخراج: {accuracy:.2f}%")

if accuracy < 100:
    print("\nلیست مغایرت‌های پیدا شده (تفاوت‌های فایل دستی و کلاود):")
    mismatches = comparison[comparison['Is_Match'] == False]
    print(mismatches)

# ذخیره نتیجه برای بررسی دقیق‌تر
comparison.to_excel('final_accuracy_report.xlsx', index=False)

--- گزارش نهایی مقایسه ---
تعداد کل روزهای مقایسه شده: 365
تعداد مطابقت‌های صحیح: 360
دقت نهایی استخراج: 98.63%

لیست مغایرت‌های پیدا شده (تفاوت‌های فایل دستی و کلاود):
     Month   Day Value_Man  Value_Cld  Is_Match
95     4.0   6.0      0.44       0.36     False
98     4.0   9.0      0.36       0.44     False
105    4.0  16.0      0.31       0.36     False
217    8.0   6.0       0.2       0.90     False
332   11.0  29.0      9.32       9.38     False


In [ ]:
import base64
import json
import re
import pandas as pd
import os
from dotenv import load_dotenv
from anthropic import Anthropic

# ۱. تنظیمات اولیه
load_dotenv()
client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

def extract_single_table(image_path):
    image_data = encode_image(image_path)
    
    # پرومپت جدید و بسیار سخت‌گیرانه برای حل مشکل Catanzaro
    system_instruction = """You are a specialized Hydrological Data Auditor. 
    Your ONLY goal is 100% digit accuracy for Catanzaro/Genova style tables.

    ### 1. THE BRACKET PROTOCOL (MANDATORY):
    - Many values are formatted like [84.10] or [12.5]. 
    - The left bracket '[' often merges with the first digit (especially digits 8, 1, and 3).
    - ACTION: Mentally 'erase' the brackets first. If you see a shape like '[8', it IS the digit 8. Do not skip the first digit.
    - Example: [84.10] must be extracted as 84.10, NOT 4.10.

    ### 2. DECIMAL PRECISION:
    - Historical prints use tiny dots. If a number looks like '865' but the context of the column suggests a small flow, it is likely '8.65'. 
    - Most values in this yearbook have ONE or TWO decimal places. Look closely for the faint dot.

    ### 3. FONT CHALLENGES:
    - **Catanzaro 1940 Special:** The ink is heavy. Digits 0, 6, 8, and 9 can look closed. Compare the 'hole' size to tell them apart.
    - **Italic Columns:** For Luglio to Dicembre, the digits lean. A '7' might look like a '1'. Check the top bar of the '7'.

    ### 4. OUTPUT:
    - Return ONLY a raw JSON array of 31 objects. 
    - Keys: "Giorno", "Gen", "Feb", "Mar", "Apr", "Mag", "Giu", "Lug", "Ago", "Set", "Ott", "Nov", "Dic"."""

    message = client.messages.create(
        model="claude-sonnet-4-6", 
        max_tokens=4000, 
        temperature=0,
        system=system_instruction,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "image", "source": {"type": "base64", "media_type": "image/gif", "data": image_data}},
                    {"type": "text", "text": "Extract all 31 days. Pay extreme attention to brackets [ ] and bold digits."}
                ],
            }
        ],
    )
    return message.content[0].text

# --- اجرای تست روی فایل مشکل‌دار ---
target_file = "../test_pictures/Q_Catanzaro_1940_54.gif"
print(f"Starting specialized extraction for: {target_file}")

try:
    raw_result = extract_single_table(target_file)
    json_match = re.search(r'\[\s*\{.*\}\s*\]', raw_result, re.DOTALL)
    
    if json_match:
        json_data = json.loads(json_match.group(0))
        df = pd.DataFrame(json_data)
        
        # ذخیره فایل جدید برای تست
        output_name = "Q_Catanzaro_1940_54_claude.xlsx"
        df.to_excel(output_name, index=False)
        print(f"✅ Finished! Result saved as {output_name}. Now run your Reformat and Accuracy codes on this file.")
    else:
        print("❌ Could not find JSON in response.")
except Exception as e:
    print(f"❌ Error: {e}")

Starting specialized extraction for: ../test_pictures/Q_Catanzaro_1940_54.gif
✅ Finished! Result saved as Q_Catanzaro_1940_54_claude.xlsx. Now run your Reformat and Accuracy codes on this file.


In [41]:
import pandas as pd
import numpy as np

# ۱. بارگذاری فایل خروجی کلاود
file_name = "Q_Catanzaro_1940_54_claude.xlsx"
df_claude = pd.read_excel(file_name)

# ۲. نقشه تبدیل ماه‌ها
month_map = {
    'Gen': 1, 'Feb': 2, 'Mar': 3, 'Apr': 4, 'Mag': 5, 'Giu': 6,
    'Lug': 7, 'Ago': 8, 'Set': 9, 'Ott': 10, 'Nov': 11, 'Dic': 12
}

# ۳. تبدیل جدول از حالت پهن به بلند (Melt)
# فرض شده ستون اول نامش 'Giorno' است
df_melted = df_claude.melt(id_vars=['Giorno'], var_name='Month_Name', value_name='Value')

# ۴. ساخت دیتای نهایی
df_final = pd.DataFrame()
df_final[0] = [1940] * len(df_melted) # ستون اول: سال
df_final[1] = df_melted['Month_Name'].map(month_map) # ستون دوم: ماه
df_final[2] = df_melted['Giorno'] # ستون سوم: روز
df_final[3] = pd.to_numeric(df_melted['Value'], errors='coerce').fillna(0) # ستون چهارم: مقدار

# مرتب‌سازی بر اساس ماه و روز
df_final = df_final.sort_values(by=[1, 2])

# ۵. ذخیره برای مقایسه
output_reform = "Q_Catanzaro_1940_54_reform.xlsx"
df_final.to_excel(output_reform, index=False, header=False)

print(f"✅ فایل ریفرم شده با نام '{output_reform}' ذخیره شد.")

✅ فایل ریفرم شده با نام 'Q_Catanzaro_1940_54_reform.xlsx' ذخیره شد.


In [44]:
import pandas as pd
import numpy as np

# ۱. بارگذاری فایل‌ها
man_path = "../test_pictures/Q_Catanzaro_1940_54.xlsx"
ref_path = "Q_Catanzaro_1940_54_reform.xlsx"

df_manual = pd.read_excel(man_path, header=None)
df_reform = pd.read_excel(ref_path, header=None)

# ۲. انتخاب ستون‌های ماه، روز و مقدار (۱ تا ۳)
df_man_3 = df_manual.iloc[:, 1:4].copy()
df_ref_3 = df_reform.iloc[:, 1:4].copy()

df_man_3.columns = ['Month', 'Day', 'Value_Man']
df_ref_3.columns = ['Month', 'Day', 'Value_Ref']

# ۳. تبدیل مقادیر به عدد و جایگزینی NaN با 0
df_man_3['Value_Man'] = pd.to_numeric(df_man_3['Value_Man'], errors='coerce').fillna(0)
df_ref_3['Value_Ref'] = pd.to_numeric(df_ref_3['Value_Ref'], errors='coerce').fillna(0)

# ۴. فیلتر کردن فایل دستی برای سال ۱۹۴۰ (ستون ۰)
df_man_1940 = df_man_3[df_manual.iloc[:, 0] == 1940].copy()

# ۵. ادغام و مقایسه
comparison = pd.merge(df_man_1940, df_ref_3, on=['Month', 'Day'])

# محاسبه مطابقت با تلورانس اعشاری
comparison['Is_Match'] = np.isclose(
    comparison['Value_Man'].astype(float), 
    comparison['Value_Ref'].astype(float), 
    atol=1e-3
)

total = len(comparison)
matches = comparison['Is_Match'].sum()
accuracy = (matches / total) * 100

print(f"--- گزارش دقت Catanzaro 1940 ---")
print(f"تعداد کل روزها: {total}")
print(f"تعداد مطابقت: {matches}")
print(f"دقت نهایی: {accuracy:.2f}%")

if accuracy < 100:
    print("\nلیست برخی اشتباهات باقی‌مانده:")
    print(comparison[comparison['Is_Match'] == False].head(10))

--- گزارش دقت Catanzaro 1940 ---
تعداد کل روزها: 366
تعداد مطابقت: 292
دقت نهایی: 79.78%

لیست برخی اشتباهات باقی‌مانده:
    Month  Day  Value_Man  Value_Ref  Is_Match
9       1   10       2.85      84.10     False
10      1   11      84.10      73.10     False
28      1   29       9.13       9.19     False
35      2    5       6.61       6.01     False
36      2    6       6.65       0.65     False
40      2   10       5.00       9.27     False
42      2   12       7.60       7.00     False
45      2   15       6.54       0.54     False
46      2   16       6.31       0.31     False
59      2   29       5.05       0.00     False
